<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Synonyms and query rewriting

**[Synonyms and query rewriting](https://learning.nextia-ai.com/courses/search/m04/synonyms-and-query-rewriting/)** · Search and Ranking: Find the Best Match · Module 4, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write one-way and two-way synonym rules, test a rule alone on the queries that it changes, mine pairs of queries from the query log, and compare a language model's recorded rewrites by language.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 11 MB). No account, no key and no model download. |
| **You should know** | Spelling correction, from [Fix spelling](https://learning.nextia-ai.com/courses/search/m04/fix-spelling/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m04/synonyms-and-query-rewriting/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M04-L03-synonyms-and-query-rewriting/synonyms-and-query-rewriting-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 11 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/log/searches.csv.gz": "73da1fb11cebb3891cabca5ab320be2147d1855b9f8e53890007b19fb78862ea",
    "data/log/impressions.csv.gz": "588cacc6c4ab6ac226478013644bf64871c69c32650fefd0219df431a38d80d0",
    "data/synonyms.csv": "bdfc3f605f0e062439a4561d1f427254cb83d0b0963b99fa4f49bbb57b075fdf",
    "data/recorded/llm_rewrites.jsonl": "a7440089eec48d8ea27f6c05e1584e28762cf428b5a81cfe78ede09cf3af80d2",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A search to test rules with

The next cell builds one BM25 index of the title and description, normalised and stemmed as in the lesson *Normalise text*. It takes less than a minute. `scores(text)` gives the BM25 score of every product for a text.

In [ ]:
import json, re, unicodedata
import bm25s, Stemmer
EN = Stemmer.Stemmer("english")

def plain(text):
    return [t for t in re.split(r"[^\w]+|_", str(text).lower()) if t]

def fold(text):
    t = unicodedata.normalize("NFKD", str(text).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"(\d)\s*(?:''|\"|-?inch(?:es)?\b|in\b)", r"\1 inch ", t)

def norm(text):
    return EN.stemWords(plain(fold(text)))

cat = catalogue()
index = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
index.index([norm(t + " " + d) for t, d in zip(cat["title"], cat["description"])], show_progress=False)
ids = cat["product_id"].to_numpy()

def scores(text):
    terms = [w for w in norm(text) if w in index.vocab_dict]
    return index.get_scores(terms) if terms else np.zeros(len(ids), dtype=np.float32)

def top10(s):
    order = np.argsort(-s)[:10]
    return [ids[i] for i in order if s[i] > 0]
print("indexed", len(ids), "products;", "synonym rules:", len(pd.read_csv("data/synonyms.csv")))

## 2. Two kinds of rule

A **two-way** rule (`couch, sofa`) adds each word to a query that has the other. A **one-way** rule (`sectional => sofa`) adds the right word to a query with the left word, never the other way.

> **Predict.** What does `couch, sofa` add to "modern sofa"? What does `sectional => sofa` add to it?

In [ ]:
def parse(rule):
    """'a, b' -> two-way: a adds b, b adds a.  'a => b' -> one-way: a adds b."""
    if "=>" in rule:
        left, right = rule.split("=>")
        return [([left.strip()], [right.strip()])]
    terms = [t.strip() for t in rule.split(",")]
    return [([t], [o for o in terms if o != t]) for t in terms]

def has(text, word):
    return re.search(rf"\b{re.escape(word)}\b", fold(text)) is not None

def added_words(text, rule):
    """The words that the rule adds to this query ('' when it does not match)."""
    out = []
    for lhs, rhs in parse(rule):
        if any(has(text, w) for w in lhs):
            out += [w for w in rhs if not has(text, w)]
    return " ".join(out)

for text in ["couch", "modern sofa", "sectional", "light blue pillow"]:
    print(f"{text:18} couch, sofa: {added_words(text, 'couch, sofa')!r:9} sectional => sofa: {added_words(text, 'sectional => sofa')!r}")

> **Check.** The two-way rule adds "couch" to "modern sofa"; the one-way rule adds nothing. A rule knows words, not meanings: "light blue pillow" would get "lamp" from a rule `light, lamp`.

## 3. Your turn: test one rule alone

Test the rule `patio, outdoor` on every validation query that it changes. The added words count half: the score with the rule is the score of the person's words plus 0.5 × the score of the added words. Complete `with_rule`.

In [ ]:
q = queries()
val = q[q["split"] == "validation"]
j = judgments()

def test_rule(rule, split="validation", weight=0.5):
    qs = q[q["split"] == split]
    rows = []
    for qid, text in zip(qs["query_id"], qs["query"]):
        extra = added_words(text, rule)
        if not extra or qid not in j:
            continue
        s = scores(text)
        with_rule = ...   # the person's words in full, plus weight x the scores of the added words
        if with_rule is ...:
            return None
        rows.append((qid, text, extra, cp.ndcg(top10(s), j[qid]), cp.ndcg(top10(with_rule), j[qid])))
    return pd.DataFrame(rows, columns=["query_id", "query", "added", "without", "with"]).round(3)

patio = test_rule("patio, outdoor")
patio_with = None
if patio is not None:
    print(patio.to_string(index=False))
    patio_with = patio["with"].mean()
    print("mean without", round(patio["without"].mean(), 3), " with", round(patio_with, 3))

In [ ]:
expect('mean NDCG@10 with the rule', patio_with, 0.651, tolerance=0.002)

> **Check.** The rule touches 6 validation queries: mean NDCG@10 from 0.636 to 0.651, 3 better and 2 worse. Read the rows, not only the mean: one rule can help one query and hurt another.

## 4. Confirm on the test queries

You tested the rule on validation queries. Before you keep it, confirm it on the test queries, which you did not look at. Then try the one-way rule `outdoor => patio`.

> **Predict.** "outdoor" is wider than "patio". Which query could the two-way rule hurt on the test split?

In [ ]:
if test_rule("patio, outdoor") is None:
    print("Not yet: complete test_rule above first.")
else:
    on_test = test_rule("patio, outdoor", split="test")
    print(on_test.to_string(index=False))
    one_way = test_rule("outdoor => patio", split="test")
    print("on the test queries: two-way", round(on_test["without"].mean(), 3), "->", round(on_test["with"].mean(), 3),
          "  one-way outdoor => patio:", round(one_way["without"].mean(), 3), "->", round(one_way["with"].mean(), 3),
          "on", len(one_way), "queries")

> **Check.** On the 7 test queries, the two-way rule goes from 0.378 to 0.421 on average. But "patio design" falls from 0.301 to 0.078: the rule adds the wider word "outdoor". The one-way rule `outdoor => patio` never touches "patio design": on its 6 test queries, from 0.391 to 0.479. The validation queries did not contain the query that the rule hurts most. That is why you confirm on queries you did not look at.

## 5. Rules mined from the log

Two different queries that lead people to click the same products probably mean the same thing. The next cell collects the clicked products of every query text in the synthetic log.

In [ ]:
log = pd.read_csv("data/log/searches.csv.gz", keep_default_na=False)
imp = pd.read_csv("data/log/impressions.csv.gz").merge(log[["search_id", "query"]], on="search_id")
clicked = imp[imp["clicked"] == 1].groupby("query")["product_id"].apply(set)
clicked = clicked[clicked.map(len) >= 3]                 # queries with at least 3 clicked products
print(len(clicked), "query texts with 3 or more clicked products (synthetic log)")

## 6. Your turn: the Jaccard similarity

The **Jaccard similarity** of two sets is the number of items in both, divided by the number of items in either. Write `jaccard`; the cell then keeps every pair of query texts with a Jaccard of 0.3 or more.

In [ ]:
def jaccard(a, b):
    """Products in both sets divided by products in either set."""
    return ...   # use a & b (in both) and a | b (in either)

print(jaccard({1, 2, 3, 4}, {2, 3, 4, 5}))
n_pairs = None
if jaccard({1}, {1}) is not ...:
    items = list(clicked.items())
    pairs = []
    for i in range(len(items)):
        for k in range(i + 1, len(items)):
            (qa, pa), (qb, pb) = items[i], items[k]
            if fold(qa) != fold(qb) and jaccard(pa, pb) >= 0.3:
                pairs.append((qa, qb, round(jaccard(pa, pb), 3)))
    pairs.sort(key=lambda r: -r[2])
    n_pairs = len(pairs)
    print(n_pairs, "pairs"); print(pairs[:8])

In [ ]:
expect('jaccard({1, 2, 3, 4}, {2, 3, 4, 5})', jaccard({1, 2, 3, 4}, {2, 3, 4, 5}), 0.6)

In [ ]:
expect('n_pairs', n_pairs, 29)

> **Check.** 29 pairs, and they are not synonyms: almost every pair is a query and the same query without its last word. Our code made the synthetic log that way (a simulated person who searches again drops the last word). Mined rules show how the log was made; with a real log, check each one before you use it.

## 7. A language model that rewrites the query

The course recorded the rewrites of **chat-small**, a small chat model on Azure, for every validation and test query. They are replayed here: no key, no cost. Compare the typed query and the rewrite, by language.

In [ ]:
rec = {r["query_id"]: r["rewrite"] for r in map(json.loads, open("data/recorded/llm_rewrites.jsonl"))}
vq = val.set_index("query_id")
rows = []
for qid, text in vq["query"].items():
    if qid in j and qid in rec:
        rows.append((qid, vq.loc[qid, "language"], cp.ndcg(top10(scores(text)), j[qid]),
                     cp.ndcg(top10(scores(rec[qid])), j[qid])))
rw = pd.DataFrame(rows, columns=["query_id", "language", "typed", "rewrite"])
print(rw.groupby("language")[["typed", "rewrite"]].mean().round(3))
allq = queries().set_index("query_id")
french = [i for i in vq.index if i.startswith("F-") and i in rec]
same_as_english = sum(rec[i].strip().lower() == allq.loc[allq.loc[i, "base_query_id"], "query"].strip().lower()
                      for i in french)
print(same_as_english, "of", len(french), "French rewrites are word for word the English query they were translated from")

> **Check.** French goes from about 0.099 to about 0.683; English from 0.570 to 0.596. But 20 of the 30 French rewrites are word for word the English query that our French query was machine-translated from. Translating machine-made French back is unrealistically easy. A live rewrite also takes about 1.2 seconds per query.

In the project: `python rule_test.py "patio, outdoor"` (from the lesson) and `search compare bm25-norm+spell+syn llm-rewrite --by source`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Detect what the query asks for](https://learning.nextia-ai.com/courses/search/m04/detect-what-the-query-asks-for/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m04/synonyms-and-query-rewriting/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.